# VL08 - Sequence models with RNN & LSTM
In this seminar, we extend our study of language models beyond ngrams and feed-forward
networks. RNNs and LSTMs are designed to process **sequential data**, making
them better suited for modelling text, where each token depends on the
previous ones.

We will reuse the same preprocessed dataset from the previous sessions and
focus on how to **define**, **train**, **use**, and **evaluate** recurrent
language models.

In summary, in this lab we will:

- build a small RNN language model from scratch,
- train it on variable-length sentences without batching,
- generate text and predict next tokens,
- upgrade the architecture to an LSTM,
- compare RNN vs LSTM behaviour and performance,
- compute cross-entropy and perplexity on a held-out test set.

By the end, you will have a complete workflow for training and analysing
recurrent neural language models.

In [ ]:
import spacy
from datasets import load_dataset, load_from_disk
import pandas as pd

# load the small English model
nlp = spacy.load("en_core_web_sm")

## 1. Preparing the dataset
We will use the a sample of articles from wikipedia. In particular, the `Salesforce/wikitext` dataset from Huggingface. If you don't have access to internet from your notebook, use the script `download_dataset.py` and then `load_from_disk()`

To download:

````bash
$ python scripts/download_dataset.py "Salesforce/wikitext" "wikitext-2-raw-v1" data/wikitext-2-raw-v1
````

In [ ]:
#ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
ds = load_from_disk("../../data/wikitext-2-raw-v1")
train_texts = ds["train"]["text"]  # list of strings
eval_texts = ds["test"]["text"]

### 1.1 Preprocessing the dataset
The Wikipedia articles in our dataset contain markup such as headings (`=== Section ===`), templates (`{{ ... }}`), or links (`[[link]]`).  
We remove these artifacts to get plain text sentences suitable for tokenization.

In [ ]:
%%time
import re
from nltk.lm import Lidstone, MLE  # good default; try KneserNeyInterpolated too
from nltk.lm.preprocessing import padded_everygram_pipeline

# 2) Cleaning wiki pages from artefacts
def clean_wikitext(lines):
    """Remove obvious wikitext artifacts."""
    for t in lines:
        t = t.strip()
        if not t:
            continue                          # skip blank lines (caused your <s>,</s> spike)
        if re.match(r"^=+\s.*\s=+$", t):
            continue                          # skip section headings like "== History =="
        # Strip templates/links (conservative to avoid nuking content)
        t = re.sub(r"\{\{.*?\}\}", "", t)     # {{ ... }}
        t = re.sub(r"\[\[|\]\]", "", t)       # [[link]] -> link
        yield t

# 2) Build a minimal pipeline: tokenizer + sentencizer (no tagger/ner/parser)
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")  # rule-based sentence boundaries

def tokenize(doc):
    # If input is a string, run spaCy tokenization first
    if isinstance(doc, str):
        doc = nlp.make_doc(doc)
    
    # Now doc is guaranteed to be a spaCy Doc or Span
    return [tok.text.lower() for tok in doc if tok.is_alpha]

# 3) Batched processing for speed
def to_sents(lines, batch_size=1000):
    sents = []
    for doc in nlp.pipe((t for t in lines if t and t.strip()), batch_size=batch_size):
        for s in doc.sents:
            toks = tokenize(s)
            if toks:
                sents.append(["<s>"] + toks + ["</s>"])
    return sents

clean_text = clean_wikitext(train_texts) 
sents_all = to_sents(clean_text)

print(sents_all[-1])

We create a smaller dataset `sents_small` to control the size of the dataset and size of the documents (number of tokens in a sentence).

In [ ]:
N_SENTENCES = 10_000
MIN_LENGTH = 5  # tokens
MAX_LENGTH = 20 # tokens

sents_small = [s for s in sents_all if MIN_LENGTH <= len(s) <= MAX_LENGTH][:N_SENTENCES]

print("sents_all ", len(sents_all), "sentences")
print("sents_small ", len(sents_small), "sentences")

### 1.2 Building the vocabulary
We build a **vocabulary** of all words in the training set, as well as an index that will allow us to encode sentences as an index. 

The resulting a vocabulary index will look something like this:

`word2idx = {'<unk>':0, 'the':1, 'german':2, …, 'war':1234}`

We can constrain the size of the vocabulary, by indexing only the tokens that appear at last `min_freq` times. This can help use reduce the number of parameter of the model (and computational requirements). This means that the words not indexed will be mapped to a special token `<unk>`.

In [ ]:
from collections import Counter

counter = Counter(tok for s in sents_small for tok in s)
# keep only frequent words to limit vocab size
min_freq = 3
vocab = ["<unk>"] + [w for w,c in counter.items() if c >= min_freq]

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

print ("original vocab ", len(counter.items()))
print ("final vocab ", len(vocab))
print()
# check if the index is working
print ("index of <unk>: ", word2idx["<unk>"] )
print ("workd at [0]: ", idx2word[0])

## 2. Recurrent Neural Networks

To model sequences like text, we use a **Recurrent Neural Network (RNN)**: a neural layer that processes one token at a time while keeping a **hidden state** containing information about previous tokens.

At each step \(t\), the RNN updates its memory:

$$
h_t = \tanh(W x_t + U h_{t-1})
$$

- $(x_t)$: embedding of the current token  
- $(h_{t-1})$: previous hidden state  
- $(h_t)$: updated “memory”  

The RNN outputs a hidden vector for every time step, which we map to a vocabulary distribution to predict the next word.

In [ ]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

### 2.1 Encoding sentences (no batching)

Our sentences have **varying length**. In this seminar, we keep things simple:

- We **do not use mini-batches**.
- We encode **one sentence at a time** and feed it directly to the model.
- This means we can work with variable-length sequences **without any padding**.

In practical setups, we usually want batching for efficiency. Then we would need:

- a fixed tensor shape `(batch_size, seq_len)`,
- padding shorter sentences up to a chosen `seq_len`,
- and a `Dataset` + `DataLoader` to build these batches.

However, to keep the focus on the **core RNN idea** (hidden state, sequence processing, language modeling), we skip batching here and train on individual sentences.


In [ ]:

def encode_tokens(tokens):
    return [word2idx.get(tok, word2idx["<unk>"]) for tok in tokens]
    
encoded_sents = []
for sent in sents_small:
    ids = encode_tokens(sent)
    ids = torch.tensor(ids, dtype=torch.long)    
    encoded_sents.append(ids)

print (sents_small[-1])
print (encoded_sents[-1])

### 2.1 Defining a single-layer RNN
We now define our RNN language model in PyTorch. The structure is very similar to the feed-forward network we used before:

- `nn.Embedding` maps token IDs to dense vectors.
- `nn.RNN` is our **single recurrent hidden layer** that processes the sequence and maintains the hidden state over time.
- `nn.Linear` maps each hidden state to a vector of logits over the vocabulary (one next-word distribution per time step).

The main difference from the FFNN is that `nn.RNN` takes the whole sequence and a previous hidden state `h` and returns:
- the sequence of hidden states (`out`)
- the final hidden state (`h`) that can be reused for continuing the sequence.

Another important aspect not visible in the architecture, is that the Pytorch's nn.RNN internally loops over the sequence, **unrolling** the cells to the sequence length (T) during the forward pass. That is:

`
h_0 → h_1 → h_2 → ... → h_T
`

During this forward pass, PyTorch also builds the computational graph across all time steps.

In [ ]:
import torch.nn as nn

class TinyRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        emb = self.embed(x)
        out, h = self.rnn(emb, h) # out: all the hi;  h: the latest h 
        logits = self.fc(out)
        return logits, h

Wait! Before moving on, we can make a small but powerful modification to our simple RNN model: **weight tying**.

Weight tying means that we **share** the parameters of the embedding matrix $(E)$ (used to map token IDs to embedding vectors) with the parameters of the output projection layer (used to map hidden states back into vocabulary logits).

In other words:

> **The embedding lookup matrix and the output layer weights become the same tensor.**

This reduces the number of parameters, often improves perplexity, and is used in many modern language models.

Implementing weight tying in PyTorch is surprisingly simple:  
we simply assign the output layer’s weight matrix to be the same tensor as the embedding weight matrix.

**Important note:**  
When tying weights, we typically **remove the bias term** from the output linear layer.  Otherwise, we would introduce additional parameters that are *not* shared, which breaks the symmetry and defeats the purpose of weight tying.

Below we redefine our simple RNN with an optional `tie_weights=True` argument.

In [ ]:
import torch.nn as nn
import torch

class TinyRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, tie_weights=False):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)

        # Output projection (remove bias if tying)
        # why? embeddings have no bias, so using + b would introduce a
        #      parameter that is not shared
        self.fc = nn.Linear(hidden_dim, vocab_size, bias=not tie_weights)

        self.tie_weights = tie_weights

        if tie_weights:
            # Enforce dimension constraint
            if hidden_dim != embed_dim:
                raise ValueError(
                    f"Weight tying requires hidden_dim == embed_dim, "
                    f"but got hidden_dim={hidden_dim}, embed_dim={embed_dim}"
                )

            # Tie weights: share parameters
            self.fc.weight = self.embed.weight

    def forward(self, x, h=None):
        emb = self.embed(x)               
        out, h = self.rnn(emb, h)
        logits = self.fc(out)
        return logits, h


### 2.2 Initialising components
We now set up the pieces needed to train our RNN language model:

- **Model**: an instance of `TinyRNN` with the vocabulary size.
- **Loss function**: `CrossEntropyLoss`, standard for next-word prediction.
- **Optimizer**: Adam, which adjusts all model parameters during training.

Then we move the model to GPU if available, otherwise CPU. In doing this, we want to put the parameters in the device memory, and later on, we will make sure that model and training data are in the same memory. 

With this setup, we are ready to run the training loop.


In [ ]:
import torch.nn as nn

model_rnn = TinyRNN(vocab_size=len(word2idx))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=1e-3)

model_rnn.to(device)

### 2.3 Training loop

We train the model by processing **one full sentence at a time**, allowing the RNN to handle **variable-length sequences** without padding. Each forward pass runs the RNN over the entire sentence, and the backward pass performs **backpropagation through time**, updating all parameters based on all time steps.

During training we:
- shuffle the sentence order each epoch,
- move tensors to the correct device (CPU/GPU),
- feed `<s>, w₁, …, wₙ₋₁` as input and predict `w₁, …, wₙ, </s>`,
- accumulate the loss over the whole sequence,
- call `loss.backward()` to propagate gradients through every step of the unrolled RNN.

**Note**: Adjust the dataset size if it takes too long.

In [ ]:
import random

EPOCHS = 10

def train_lm(model, encoded_sents, optimizer, criterion, device, epochs=10):
    """
    Train a language model (RNN or LSTM) on encoded sentences.

    - Processes ONE sentence at a time (no batching, no padding).
    - Uses teacher forcing for next-word prediction.
    - Works for both TinyRNN and TinyLSTM (same interface).
    """
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        used_sents = 0

        # Shuffle sentences each epoch so the model does not always
        # see them in the same order
        random.shuffle(encoded_sents)

        for ids in encoded_sents:
            # Skip very short sentences (< 2 tokens)
            if ids.size(0) < 2:
                continue

            # Make sure data lives on the same device as the model
            ids = ids.to(device)

            # input:  <s>, w1, ..., w_{N-1}  
            # target: w1,  w2, ..., w_N
            x = ids[:-1].unsqueeze(0)   # (batch=1, seq_len) format expected as input
            y = ids[1:]                 # (seq_len,)

            optimizer.zero_grad()

            # Forward pass over the whole sequence
            # For RNN:  h_T
            # For LSTM: (h_T, c_T)   → we don't need them here
            logits, _ = model(x)        # (1, seq_len, vocab_size)
            logits = logits.squeeze(0)  # (seq_len, vocab_size)

            # Loss summed over all time steps in the sentence
            loss = criterion(logits, y)

            # Backpropagation through time happens here
            loss.backward()  # compute gradients
            optimizer.step() # update weights / parameters

            total_loss += loss.item()
            used_sents += 1

        avg_loss = total_loss / max(used_sents, 1)
        print(f"Epoch {epoch+1}: avg loss = {avg_loss:.4f}")

In [ ]:
train_lm(model_rnn, encoded_sents, optimizer, criterion, device, epochs = 10)

### 2.4 Next-token prediction

Now that our model is trained, we can start testing how it performs. Unfortunately, our model does not have a built-in `predict` function, but we can easily create one.  
To predict the next word:

1. We take a text prefix and tokenize it.
2. We prepend the `<s>` token to match the training format.
3. We run a forward pass of the RNN to obtain logits for every position.
4. We keep only the logits of the **last time step**, since they correspond to the next-word distribution.
5. We apply a softmax to convert logits into probabilities.
6. We use PyTorch’s `topk` to extract the most likely next words.

This gives us an interpretable next-token prediction that mirrors what we did with N-grams, but now using a neural language model.


In [ ]:
import torch.nn.functional as F

# we get the index in the vocab, of start of sentence 
SOS = word2idx["<s>"]

# We use previously defined tokenize and encode_tokens

def topk_next_lm(model, prefix, k=10):
    """
    prefix: string or list of tokens (without <s>, </s>)
    k: how many likely next tokens to show
    """
    model.eval()

    # normalise input
    tokens = tokenize(prefix)

    # encode with <s> at start
    ids = [SOS] + encode_tokens(tokens)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, seq_len)

    with torch.no_grad():
        logits, _ = model(x)                     # (1, seq_len, vocab)
        # we get the last logits, batch 0, and last token in the sequence (-1)
        last_logits = logits[0, -1]              # (vocab,)
        probs = F.softmax(last_logits, dim=-1)   # (vocab,)
        print(prefix, "-->")
        topk = torch.topk(probs, k)
        for idx, p in zip(topk.indices, topk.values):
            print(f"{idx2word[idx.item()]:<15} {p.item():.4f}")
        print()

In [ ]:
topk_next_lm(model_rnn, "In", k=10)
topk_next_lm(model_rnn, "In the", k=10)
topk_next_lm(model_rnn, "As a result", k=10)

### 2.5 Sentence generation

Our RNN is a sequence model, so we can generate text by repeatedly sampling the next token.  
There is no built-in `generate` function, so we implement a minimal version:

- Start from the `<s>` token.
- Run one forward step to get the next-token distribution.
- Sample a word from this distribution.
- Feed it back into the model.
- Stop when we reach `</s>` or a maximum length.

This simple loop is enough to produce short, locally coherent sequences that reflect what the model has learned.


In [ ]:
import math

EOS = word2idx["</s>"] # index of the start of the sentence

def generate_lm(model, max_len=25):
    """
    Generate a sentence from the trained language model (RNN / LSTM).

    - Starts from <s>
    - At each step: predict next-word distribution and sample one token
    - Stops when </s> is produced or max_len is reached
    """
    model.eval()
    h = None              # initial hidden state (RNN will start with zeros)
    idx = SOS             # start from <s>
    generated = []        # list of token strings

    for _ in range(max_len):
        # current input is a single token ID, shaped as (batch=1, seq_len=1)
        x = torch.tensor([[idx]], dtype=torch.long, device=device)

        with torch.no_grad():
            # forward one step: use previous hidden state h
            logits, h = model(x, h)   # logits: (1, 1, vocab_size)

        # convert logits of the last (and only) time step to probabilities
        probs = F.softmax(logits[0, -1], dim=-1) # dim=-1 means last dimension (vocab_size)

        # sample a token ID according to the probability distribution
        idx = torch.multinomial(probs, num_samples=1).item()

        # stop if we generated the end-of-sentence token
        if idx == EOS:
            break

        # store the generated word
        generated.append(idx2word[idx])

    # join all generated tokens into a single string
    return " ".join(generated)


In [ ]:
for _ in range(5):
    print(generate_lm(model_rnn, max_len=30))
    print()

## 3. LSTM

Recurrent Neural Networks can, in practice, struggle with **longer dependencies** because gradients tend to vanish or explode over many time steps.  
Long Short-Term Memory (LSTM) networks address this by introducing a more sophisticated recurrent cell with an internal **cell state** and **gates** that control what to keep, forget, and output.

In this section, we reuse the **same encoded data and training setup** as for the RNN. The only change is the **architecture of the recurrent layer**.


### 3.1 Defining the model

The LSTM language model has the same overall structure as our TinyRNN:

- an `nn.Embedding` layer to map token IDs to vectors,
- a **single recurrent hidden layer**, now implemented with `nn.LSTM` instead of `nn.RNN`,
- a final `nn.Linear` layer that maps hidden states to vocabulary logits.

The main difference is that `nn.LSTM` maintains **two states**:
- the hidden state $(h_t)$ (like in a standard RNN),
- the cell state $(c_t)$, which carries longer-term information.

Apart from that, the forward method and the rest of the training code stay almost identical to the RNN version.


In [ ]:
import torch.nn as nn

class TinyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        """
        x: (batch, seq_len)
        h: tuple (h_0, c_0) or None
           - h_0: initial hidden state
           - c_0: initial cell state
        """
        emb = self.embed(x)               # (batch, seq_len, embed_dim)
        out, h = self.lstm(emb, h)        # out: (batch, seq_len, hidden_dim)
                                          # h: (h_T, c_T)
        logits = self.fc(out)             # (batch, seq_len, vocab_size)
        return logits, h

### 3.2 Training the model
We repeat the same training setup used for the RNN model:

- create an instance of our `TinyLSTM`,
- define the loss function (`CrossEntropyLoss`),
- define the optimiser (Adam),
- move everything to the correct device.

Since the training loop logic is identical for RNNs and LSTMs, we simply reuse the `train_lm` helper defined earlier, passing in the LSTM model and the encoded sentences.

In [ ]:
import torch.nn as nn

model_lstm = TinyLSTM(vocab_size=len(word2idx)).to(device)
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_lm(model_lstm, encoded_sents, optimizer, criterion, device, epochs=10)

### 3.3 Testing the model

To evaluate the behaviour of our LSTM language model, we reuse the same helper
functions we introduced earlier:

- `topk_next_lm()` for next-token prediction,
- `generate_lm()` for sampling new sentences.

By passing the trained LSTM model to these helpers, we can directly compare its
predictions and generated text against the earlier RNN results and observe the
improvements in modelling longer-range dependencies.

In [ ]:
topk_next_lm(model_lstm, "In", k=10)
topk_next_lm(model_lstm, "In the", k=10)
topk_next_lm(model_lstm, "As a result", k=10)

In [ ]:
for _ in range(5):
    print(generate_lm(model_lstm, max_len=30))
    print()

## 4. Evaluation

To evaluate our language models, we use the same metrics introduced in the
N-gram lab: **cross-entropy** and **perplexity**.  
These metrics quantify how well a model predicts the next token in a held-out
test set.

As with training, we must process the test split using **the exact same
pipeline**: cleaning, tokenising, adding `<s>` and `</s>` markers, and encoding
into token IDs. We then run the model forward over every sentence in the test
set and accumulate the total negative log-likelihood.

The helper function below performs this procedure and returns:

- cross-entropy (nats per token),
- perplexity,
- and the number of test sentences used.



In [ ]:
criterion_eval = nn.CrossEntropyLoss(reduction="sum")  # sum over tokens

def lm_cross_entropy_perplexity_from_sents(model, sents, max_len=None, device=None):
    """
    Compute corpus cross-entropy (nats/token) and perplexity for an RNN/LSTM LM.

    sents: list of token lists, e.g. ["<s>", "in", "the", "city", "</s>"]
    max_len: optional cap on sentence length
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    total_loss = 0.0
    total_tokens = 0
    used_sents = 0

    with torch.no_grad():
        for toks in sents:
            if max_len is not None:
                toks = toks[:max_len]

            ids = encode_tokens(toks)
            if len(ids) < 2:
                continue

            ids_tensor = torch.tensor(ids, dtype=torch.long, device=device)
            x = ids_tensor[:-1].unsqueeze(0)   # (1, seq_len)
            y = ids_tensor[1:]                 # (seq_len,)

            logits, _ = model(x)               # (1, seq_len, vocab)
            logits = logits.squeeze(0)         # (seq_len, vocab)

            loss = criterion_eval(logits, y)
            total_loss += loss.item()
            total_tokens += y.size(0)
            used_sents += 1

    if total_tokens == 0:
        return math.inf, math.inf, 0

    H = total_loss / total_tokens
    PP = math.exp(H)
    return H, PP, used_sents


In [ ]:
# same process as in the N-gram lab
clean_eval = list(clean_wikitext(eval_texts))
sents_eval = to_sents(clean_eval)  # list of token lists, with <s>, </s>
len(sents_eval)

In [ ]:
short_sents  = [s for s in sents_eval if len(s) <= 10]
medium_sents = [s for s in sents_eval if 10 < len(s) <= 25]
long_sents   = [s for s in sents_eval if len(s) > 25]

# let's sample the same number
N = min(len(short_sents), len(medium_sents), len(long_sents))

short_sample  = random.sample(short_sents,  N)
medium_sample = random.sample(medium_sents, N)
long_sample   = random.sample(long_sents,   N)

print (len(short_sample))
print (len(medium_sample))
print (len(long_sample))

### 4.1 Testing our Tiny Models

In [ ]:

eval_sets = {
    "all"   : sents_eval,
    "short" : short_sample,
    "medium": medium_sample,
    "long"  : long_sample,
}

print("=== RNN vs LSTM evaluation by length bucket ===")
for name, sents in eval_sets.items():
    H_rnn, PP_rnn, used_rnn = lm_cross_entropy_perplexity_from_sents(
        model_rnn, sents, max_len=None, device=device
    )
    H_lstm, PP_lstm, used_lstm = lm_cross_entropy_perplexity_from_sents(
        model_lstm, sents, max_len=None, device=device
    )
    print(f"\nBucket: {name} (sentences used: RNN={used_rnn}, LSTM={used_lstm})")
    print(f"  RNN  → H: {H_rnn:.4f} nats/token | PP: {PP_rnn:.2f}")
    print(f"  LSTM → H: {H_lstm:.4f} nats/token | PP: {PP_lstm:.2f}")


### Things to try
1. Train longer. Does the LSTM outperform the RNN after 20–30 epochs instead of 10?
2. Increase hidden size. Does a larger hidden state (e.g., 64 instead of 32) help the LSTM more than the RNN?
3. Evaluate by sentence length. Does the gap between RNN and LSTM increase for very long sentences (e.g., >30 or >50 tokens)?

### 4.1 (Extra) Testing bigger models

To better see the benefits of LSTMs, we now train **larger** RNN and LSTM
language models on **more data**. We increase the embedding and hidden
dimensions and use more than 10,000 training sentences. With this setup, does LSTM start to show an advantage in terms of cross-entropy and perplexity?


In [ ]:
# Testing bigger models
N_EPOCS = 20
# 1) Build a larger training subset (10,000 sentences)
#    Assume `sents_all` contains all tokenised sentences (with <s>, </s>),
#    as in the earlier preprocessing step.
sents_10k = sents_all[:30_000]

# 1) Build vocab on the 10k subset
from collections import Counter

counter = Counter()
for sent in sents_10k:
    for tok in sent:
        counter[tok] += 1

# apply min_freq
min_freq = 2
vocab = ["<unk>"] + [tok for tok, c in counter.items() if c >= min_freq]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

SOS = word2idx["<s>"]
EOS = word2idx["</s>"]


# 2) Re-encode the 10k sentences
encoded_sents_10k = []
for sent in sents_10k:
    ids = [word2idx.get(tok, word2idx["<unk>"]) for tok in sent]
    if len(ids) >= 3:
        encoded_sents_10k.append(torch.tensor(ids))

# 2) Define bigger models: higher capacity than TinyRNN/TinyLSTM
embed_dim_big = 32
hidden_dim_big = 64

model_rnn_big = TinyRNN(
    vocab_size=len(word2idx),
    embed_dim=embed_dim_big,
    hidden_dim=hidden_dim_big
).to(device)

model_lstm_big = TinyLSTM(
    vocab_size=len(word2idx),
    embed_dim=embed_dim_big,
    hidden_dim=hidden_dim_big
).to(device)

# 3) Train both models with the same setup
criterion = nn.CrossEntropyLoss()

optimizer_rnn_big  = torch.optim.Adam(model_rnn_big.parameters(),  lr=1e-3)
optimizer_lstm_big = torch.optim.Adam(model_lstm_big.parameters(), lr=1e-3)

print("=== Training bigger RNN model ===")
train_lm(model_rnn_big, encoded_sents_10k, optimizer_rnn_big, criterion, device, epochs=N_EPOCS)

print("\n=== Training bigger LSTM model ===")
train_lm(model_lstm_big, encoded_sents_10k, optimizer_lstm_big, criterion, device, epochs=N_EPOCS)


In [ ]:
print("=== RNN vs LSTM evaluation by length bucket ===")
for name, sents in eval_sets.items():
    H_rnn_big, PP_rnn_big, used_rnn = lm_cross_entropy_perplexity_from_sents(
        model_rnn_big, sents, device=device
    )
    H_lstm_big, PP_lstm_big, used_lstm = lm_cross_entropy_perplexity_from_sents(
        model_lstm_big, sents, device=device
    )
    
    print(f"\nBucket: {name} (sentences used: RNN={used_rnn}, LSTM={used_lstm})")
    print(f"  RNN  → H: {H_rnn_big:.4f} nats/token | PP: {PP_rnn_big:.2f}")
    print(f"  LSTM → H: {H_lstm_big:.4f} nats/token | PP: {PP_lstm_big:.2f}")

In [ ]:
print ("----RNN big----")
for _ in range(5):
    print(generate_lm(model_rnn_big, max_len=30))
    print()
print ("----LSTM big----")
for _ in range(5):
    print(generate_lm(model_lstm_big, max_len=30))
    print()